# optimizer-init-params-list composite — cx1: minimal SGD optimizer: __init__ materializes params, step() updates in-place

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `optimizer-init-params-list`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-init-params-list"
DD_ATOM_IDS = ["optimizer-init-params-list", "inplace-param-update"]
DD_SUBTOPICS = ["PyTorch: Optimizer init", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A PyTorch optimizer is a thin object that takes an iterable of `nn.Parameter`s and a learning rate, and exposes a `.step()` method that applies one update rule (SGD, Adam, ...) in-place to those params.

The TWO atoms you compose here are the minimum to build a working SGD:
1. **optimizer-init-params-list** — in `__init__`, the `params` argument is usually a GENERATOR (returned by `model.parameters()`). Generators are single-pass, so the optimizer MUST materialize it into a list (or tuple) once, up front: `self.params = list(params)`. After that, the same list is iterated by every `.step()` call.
2. **inplace-param-update** — inside `.step()`, the update is `p.data -= lr * p.grad` (or equivalently `p.data.sub_(p.grad, alpha=lr)`). Two crucial details:
   - We mutate `p.data`, not `p`, so autograd doesn't track the update as a graph op.
   - We mutate IN-PLACE (`-=`, not `p = p - ...`). A non-in-place assign would create a new tensor and the model's `p` would still point at the OLD weights.
   - The full step is wrapped in `t.no_grad()` so autograd doesn't try to build a graph through the optimizer's own arithmetic.

**Anatomy.**
```python
class SGD:
    def __init__(self, params, lr):
        self.params = list(params)            # atom A: materialize generator.
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p.data -= self.lr * p.grad        # atom B: in-place update of .data.
```

**Why both atoms together.** Without materializing the generator, the second `.step()` call would see an empty iterator and silently do nothing. Without the in-place `-=`, the model would never actually update.

### Composite Exercise — minimal SGD optimizer: __init__ materializes params, step() updates in-place

**Atoms exercised together**: `optimizer-init-params-list`, `inplace-param-update`

Implement `cx1_make_sgd()` — return a class `MySGD` (do NOT subclass `torch.optim.Optimizer`; build a from-scratch class) such that:

- `MySGD(params, lr)`:
  - Materializes `params` into `self.params = list(params)`.
  - Stores `self.lr = lr`.
- `MySGD.step(self)`:
  - For each `p` in `self.params`: if `p.grad is not None`, do `p.data -= self.lr * p.grad`.
  - Wrap the loop in `with t.no_grad():` (or decorate with `@t.no_grad()`).

The test cross-checks against `torch.optim.SGD(..., lr=0.1, momentum=0)` on the SAME starting parameters and the SAME gradient. After one `.step()`, your params must equal PyTorch's. The test also passes a GENERATOR (not a list) and calls `.step()` TWICE to verify the second call still sees the same params.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx1_make_sgd():
    """Return the MySGD class."""
    raise NotImplementedError

def _test_cx1():
    MySGD = cx1_make_sgd()
    assert isinstance(MySGD, type), 'cx1 must return a class'

    # Case A: materialize a generator (single-pass) — two steps must both update.
    t.manual_seed(0)
    p1 = t.nn.Parameter(t.randn(3, 4))
    p2 = t.nn.Parameter(t.randn(5))
    p1.grad = t.ones_like(p1)
    p2.grad = t.full_like(p2, 2.0)
    p1_before = p1.detach().clone()
    p2_before = p2.detach().clone()

    def param_gen():
        yield p1
        yield p2

    opt = MySGD(param_gen(), lr=0.1)
    opt.step()
    # After one step: p -= 0.1 * grad.
    assert t.allclose(p1.data, p1_before - 0.1 * t.ones_like(p1)), 'p1 not updated correctly after step 1'
    assert t.allclose(p2.data, p2_before - 0.1 * t.full_like(p2, 2.0)), 'p2 not updated correctly after step 1'
    # Now refresh grads and call step AGAIN — should update again (proves generator was materialized).
    p1.grad = t.ones_like(p1)
    p2.grad = t.full_like(p2, 2.0)
    opt.step()
    assert t.allclose(p1.data, p1_before - 0.2 * t.ones_like(p1)), 'second step did not update p1 — generator not materialized?'
    assert t.allclose(p2.data, p2_before - 0.2 * t.full_like(p2, 2.0)), 'second step did not update p2 — generator not materialized?'

    # Case B: cross-check vs torch.optim.SGD on identical setup.
    t.manual_seed(1)
    weight_init = t.randn(7, 3)
    grad_val = t.randn(7, 3)

    p_mine = t.nn.Parameter(weight_init.clone())
    p_mine.grad = grad_val.clone()
    opt_mine = MySGD([p_mine], lr=0.05)
    opt_mine.step()

    p_ref = t.nn.Parameter(weight_init.clone())
    p_ref.grad = grad_val.clone()
    opt_ref = t.optim.SGD([p_ref], lr=0.05, momentum=0)
    opt_ref.step()

    assert t.allclose(p_mine.data, p_ref.data, atol=1e-7), (
        f'MySGD diverges from torch.optim.SGD; max err = {(p_mine.data - p_ref.data).abs().max().item()}'
    )

    # Case C: in-place mutation — p_mine is the SAME tensor object before/after step.
    p3 = t.nn.Parameter(t.randn(4))
    p3.grad = t.ones_like(p3)
    tensor_id_before = id(p3)
    data_id_before = id(p3.data)  # tricky: data identity may stay, but VALUES must change.
    opt = MySGD([p3], lr=0.1)
    before_clone = p3.detach().clone()
    opt.step()
    assert id(p3) == tensor_id_before, 'param tensor was replaced — must be in-place'
    assert not t.allclose(p3.data, before_clone), 'p3 was not actually updated'

    # Case D: p.grad is None — must be skipped, not crash.
    p4 = t.nn.Parameter(t.randn(2))
    p4.grad = None
    p5 = t.nn.Parameter(t.randn(2))
    p5.grad = t.ones_like(p5)
    p5_before = p5.detach().clone()
    opt = MySGD([p4, p5], lr=0.1)
    opt.step()  # must not crash on p4
    assert t.allclose(p5.data, p5_before - 0.1), 'p5 should still update even when p4.grad is None'
    _dd_passed.add('cx1')

_test_cx1()

<details><summary>Show solution — cx1</summary>

```python
def cx1_make_sgd():
    class MySGD:
        def __init__(self, params, lr):
            # Atom A (optimizer-init-params-list): materialize the generator once.
            # If we kept the generator object, the second .step() would iterate an exhausted iterator.
            self.params = list(params)
            self.lr = lr

        @t.no_grad()
        def step(self):
            for p in self.params:
                if p.grad is None:
                    continue
                # Atom B (inplace-param-update): mutate .data in place; do NOT reassign p.
                p.data -= self.lr * p.grad

    return MySGD
```

Three common bugs the test catches: (1) storing the generator instead of materializing — second step silently no-ops; (2) writing `p = p - lr * p.grad` instead of `p.data -= ...` — rebinds the local name without touching the model's parameter; (3) forgetting `t.no_grad()` — the optimizer math itself enters the autograd graph and you get a slow memory leak. PyTorch's `Optimizer` base class handles (1) via `param_groups`; here we do it by hand.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["PyTorch: Optimizer init", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()